In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir("/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SafeLoRA")

In [4]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.0 MB/s eta 0:00:00


In [5]:
config_path = 'configs/llama/config.yaml'

save_folder = 'outputs/llama_alpaca/train_eval'

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
import datasets
import os
os.environ["WANDB_DISABLED"] = "true"

data_path = "/content/drive/MyDrive/CAISI_phase2/safe-finetuning/ft_datasets/alpaca_clean_sample_5k"
dataset = datasets.load_from_disk(data_path)
train_ds = dataset['train']
val_ds = dataset['test']
def convert_to_template_from_alpaca(example):
        return {"prompt": [{"role": "user", "content": example['instruction'] + ": " + example['input']}],
                "completion": [{"role": "assistant", "content": example['output']}]
        }

TEMPLATE_FUNC_MAP = {
  'alpaca': convert_to_template_from_alpaca
}

template_func = TEMPLATE_FUNC_MAP['alpaca']

og_cols = list(train_ds.column_names)
train_ds = train_ds.map(template_func, remove_columns=og_cols)
val_ds = val_ds.map(template_func, remove_columns=og_cols)

In [ ]:
# load train_ds and val_ds, run formatting with map(template_func, remove_columns=og_cols)
# load saved PEFT model, load tokenizer
model_path = "outputs/llama_alpaca/checkpoints"
base_model =  "meta-llama/Meta-Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_path,
               dtype="float16")
tokenizer = AutoTokenizer.from_pretrained(base_model)

trainer = SFTTrainer(
        model=model, # peft model
        args=SFTConfig(
            logging_dir=save_folder,
            logging_steps=10,
            log_level='info'
        ), # are not actually training so we don't care much about the args. Maybe just add the ones for logging_dir and logging_steps?
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer
    )

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
#Eval loss
trainer.evaluate() # just run the evaluation

In [ ]:
#Training loss
tokenized_train = trainer._prepare_dataset(train_ds, tokenizer=tokenizer)
trainer.evaluate(eval_dataset=tokenized_train)